## （１）国土数値情報 駅別乗降客数データ (S12-25)

In [1]:
import zipfile
from pathlib import Path

import pandas as pd
import geopandas as gpd

pd.set_option("display.max_columns", None)

# このノートブックは script/ に置かれている想定。リポジトリ直下を基準にする。
BASE_DIR = Path.cwd().parent if Path.cwd().name == "script" else Path.cwd()
ZIP_PATH = BASE_DIR / "data" / "駅別乗降客数データ" / "S12-25_GML.zip"
assert ZIP_PATH.exists(), f"ZIP が見つかりません: {ZIP_PATH}"

In [2]:
# ZIP 内のファイル一覧を確認
with zipfile.ZipFile(ZIP_PATH) as zf:
    for name in zf.namelist():
        print(name)

S12-25_GML/
S12-25_GML/KS-META-S12-25.xml
S12-25_GML/KsjAppSchema-S12-v3_3.xsd
S12-25_GML/Shift-JIS/
S12-25_GML/Shift-JIS/S12-25.xml
S12-25_GML/Shift-JIS/S12-25_NumberOfPassengers.dbf
S12-25_GML/Shift-JIS/S12-25_NumberOfPassengers.geojson
S12-25_GML/Shift-JIS/S12-25_NumberOfPassengers.prj
S12-25_GML/Shift-JIS/S12-25_NumberOfPassengers.shp
S12-25_GML/Shift-JIS/S12-25_NumberOfPassengers.shx
S12-25_GML/UTF-8/
S12-25_GML/UTF-8/S12-25.xml
S12-25_GML/UTF-8/S12-25_NumberOfPassengers.dbf
S12-25_GML/UTF-8/S12-25_NumberOfPassengers.geojson
S12-25_GML/UTF-8/S12-25_NumberOfPassengers.prj
S12-25_GML/UTF-8/S12-25_NumberOfPassengers.shp
S12-25_GML/UTF-8/S12-25_NumberOfPassengers.shx


In [3]:
# ZIP を解凍せず、geopandas の zip:// 仮想パスで直接 GeoJSON を読み込む
GEOJSON_INNER = "S12-25_GML/UTF-8/S12-25_NumberOfPassengers.geojson"
read_path = f"zip://{ZIP_PATH}!{GEOJSON_INNER}"

gdf = gpd.read_file(read_path)
print("行数 x 列数:", gdf.shape)
print("CRS:", gdf.crs)

行数 x 列数: (10534, 64)
CRS: EPSG:6668


In [4]:
# 属性列を S12 仕様の日本語名にリネーム
# 固定項目
rename_map = {
    "S12_001": "駅名",
    "S12_001c": "駅コード",
    "S12_001g": "グループコード",
    "S12_002": "運営会社",
    "S12_003": "路線名",
    "S12_004": "鉄道区分",
    "S12_005": "事業者種別",
}

# 年度ごとに [重複コード, データ有無コード, 備考, 乗降客数] が4列ずつ繰り返す
# 2011年度=S12_006〜S12_009 ... 2024年度=S12_058〜S12_061
for i, year in enumerate(range(2011, 2025)):
    base = 6 + i * 4
    rename_map[f"S12_{base:03d}"] = f"重複コード{year}"
    rename_map[f"S12_{base + 1:03d}"] = f"データ有無コード{year}"
    rename_map[f"S12_{base + 2:03d}"] = f"備考{year}"
    rename_map[f"S12_{base + 3:03d}"] = f"乗降客数{year}"

gdf = gdf.rename(columns=rename_map)
list(gdf.columns)

['駅名',
 '駅コード',
 'グループコード',
 '運営会社',
 '路線名',
 '鉄道区分',
 '事業者種別',
 '重複コード2011',
 'データ有無コード2011',
 '備考2011',
 '乗降客数2011',
 '重複コード2012',
 'データ有無コード2012',
 '備考2012',
 '乗降客数2012',
 '重複コード2013',
 'データ有無コード2013',
 '備考2013',
 '乗降客数2013',
 '重複コード2014',
 'データ有無コード2014',
 '備考2014',
 '乗降客数2014',
 '重複コード2015',
 'データ有無コード2015',
 '備考2015',
 '乗降客数2015',
 '重複コード2016',
 'データ有無コード2016',
 '備考2016',
 '乗降客数2016',
 '重複コード2017',
 'データ有無コード2017',
 '備考2017',
 '乗降客数2017',
 '重複コード2018',
 'データ有無コード2018',
 '備考2018',
 '乗降客数2018',
 '重複コード2019',
 'データ有無コード2019',
 '備考2019',
 '乗降客数2019',
 '重複コード2020',
 'データ有無コード2020',
 '備考2020',
 '乗降客数2020',
 '重複コード2021',
 'データ有無コード2021',
 '備考2021',
 '乗降客数2021',
 '重複コード2022',
 'データ有無コード2022',
 '備考2022',
 '乗降客数2022',
 '重複コード2023',
 'データ有無コード2023',
 '備考2023',
 '乗降客数2023',
 '重複コード2024',
 'データ有無コード2024',
 '備考2024',
 '乗降客数2024',
 'geometry']

In [5]:
gdf.info()
gdf.head()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 10534 entries, 0 to 10533
Data columns (total 64 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   駅名            10534 non-null  object  
 1   駅コード          10297 non-null  object  
 2   グループコード       10297 non-null  object  
 3   運営会社          10534 non-null  object  
 4   路線名           10534 non-null  object  
 5   鉄道区分          10534 non-null  int32   
 6   事業者種別         10534 non-null  int32   
 7   重複コード2011     10534 non-null  int32   
 8   データ有無コード2011  10534 non-null  int32   
 9   備考2011        122 non-null    object  
 10  乗降客数2011      10534 non-null  int32   
 11  重複コード2012     10534 non-null  int32   
 12  データ有無コード2012  10534 non-null  int32   
 13  備考2012        144 non-null    object  
 14  乗降客数2012      10534 non-null  int32   
 15  重複コード2013     10534 non-null  int32   
 16  データ有無コード2013  10534 non-null  int32   
 17  備考2013        159 non-null    object  
 18

,駅名,駅コード,グループコード,運営会社,路線名,鉄道区分,事業者種別,重複コード2011,データ有無コード2011,備考2011,乗降客数2011,重複コード2012,データ有無コード2012,備考2012,乗降客数2012,重複コード2013,データ有無コード2013,備考2013,乗降客数2013,重複コード2014,データ有無コード2014,備考2014,乗降客数2014,重複コード2015,データ有無コード2015,備考2015,乗降客数2015,重複コード2016,データ有無コード2016,備考2016,乗降客数2016,重複コード2017,データ有無コード2017,備考2017,乗降客数2017,重複コード2018,データ有無コード2018,備考2018,乗降客数2018,重複コード2019,データ有無コード2019,備考2019,乗降客数2019,重複コード2020,データ有無コード2020,備考2020,乗降客数2020,重複コード2021,データ有無コード2021,備考2021,乗降客数2021,重複コード2022,データ有無コード2022,備考2022,乗降客数2022,重複コード2023,データ有無コード2023,備考2023,乗降客数2023,重複コード2024,データ有無コード2024,備考2024,乗降客数2024,geometry
0,二月田,010112,010112,九州旅客鉄道,指宿枕崎線,11,2,1,3,None,0,1,3,None,0,1,3,None,0,1,3,None,0,1,3,None,0,1,3,None,0,1,3,None,0,1,1,None,658,1,1,None,690,1,1,None,318,1,1,None,305,1,1,None,622,1,1,None,664,1,1,None,726,"LINESTRING (130.63035 31.25405, 130.62985 31.2..."
1,古島,010127,010127,沖縄都市モノレール,沖縄都市モノレール線,23,5,1,1,None,3907,1,1,None,3980,1,1,None,4572,1,1,None,4187,1,1,None,4648,1,1,None,4528,1,1,None,4819,1,1,None,5086,1,1,None,5143,1,1,None,3404,1,1,None,3534,1,1,None,4401,1,1,None,4903,1,1,None,5490,"LINESTRING (127.70279 26.23035, 127.70309 26.2..."
2,お台場海浜公園,004091,004091,ゆりかもめ,東京臨海新交通臨海線,24,5,1,1,None,14612,1,1,None,16130,1,1,None,16357,1,1,None,16515,1,1,None,17537,1,1,None,17488,1,1,None,16944,1,1,None,17165,1,1,None,16899,1,1,None,7703,1,1,None,8535,1,1,None,11171,1,1,None,13195,1,1,None,13435,"LINESTRING (139.77818 35.62961, 139.77888 35.63)"
3,東京国際クルーズターミナル,004128,004128,ゆりかもめ,東京臨海新交通臨海線,24,5,1,1,None,3767,1,1,None,3235,1,1,None,3407,1,1,None,3782,1,1,None,3682,1,1,None,3682,1,1,None,3590,1,1,None,3551,1,1,None,3191,1,1,None,1383,1,1,None,1984,1,1,None,2300,1,1,None,2963,1,1,None,3503,"LINESTRING (139.77333 35.62109, 139.77288 35.6..."
4,テレコムセンター,004144,004144,ゆりかもめ,東京臨海新交通臨海線,24,5,1,1,None,12112,1,1,None,12775,1,1,None,13366,1,1,None,14138,1,1,None,14069,1,1,None,14069,1,1,None,13744,1,1,None,13475,1,1,None,12140,1,1,None,6536,1,1,None,7631,1,1,None,8118,1,1,None,8505,1,1,None,9267,"LINESTRING (139.78001 35.61791, 139.77932 35.6..."


## （２）駅グループ単位への集約

`docs/passenger_aggregation.md` の設計に基づき、**駅名 + 半径1km** で駅をまとめ、
レベル時系列（各年の乗降客数）と増減率（パターン1・2）を生成する。

- 在/欠測の判定: `データ有無コード==1` を実測（値0も実測の0）、それ以外は欠測（0扱いしない）
- 構成単位: 運営会社（路線は事業者内で合算。重複行は0なので二重計上なし）
- 出力ファイル（最終セル）: メイン **`station_dataset.csv`**（1群1行・人口等を統合）＋ 監査明細 **`station_operator_detail.csv`**（1群×1社）。日本語の列【名】は出力時にASCIIへ（`駅名→station_name`, `都道府県→prefecture`, `運営会社→operator`／値は日本語のまま）

In [6]:
# 集約のための定数と前処理
import numpy as np
from sklearn.cluster import DBSCAN

YEARS = list(range(2011, 2025))
PRE_WINDOW  = [2017, 2018, 2019]   # コロナ前の参照窓（平常年は代替可能）
POST_WINDOW = [2023, 2024]         # コロナ後（回復期）の参照窓。2022以前は底値なので不可

# 乗降客数列を数値化（在/欠測の判定は「データ有無コード」で行うため、ここは数値化のみ）
for y in YEARS:
    gdf[f"乗降客数{y}"] = pd.to_numeric(gdf[f"乗降客数{y}"], errors="coerce")

In [7]:
# 空間グループ: 代表点を取り、駅名ごとに haversine DBSCAN(eps=1km) でクラスタリング
_pt = gdf.geometry.representative_point()
gdf["lon"] = _pt.x
gdf["lat"] = _pt.y

EARTH_R = 6_371_000.0          # 地球半径[m]
EPS     = 1000 / EARTH_R       # 半径1km を弧度に換算

grp = pd.Series(index=gdf.index, dtype=object)
for name, sub_idx in gdf.groupby("駅名").groups.items():
    sub_idx = list(sub_idx)
    if len(sub_idx) == 1:                       # 単独駅はクラスタリング不要
        grp.loc[sub_idx[0]] = f"{name}#0"
        continue
    coords = np.radians(gdf.loc[sub_idx, ["lat", "lon"]].to_numpy())
    lab = DBSCAN(eps=EPS, min_samples=1, metric="haversine").fit(coords).labels_
    for j, l in zip(sub_idx, lab):
        grp.loc[j] = f"{name}#{l}"
gdf["grp"] = grp   # 例: "東京#0"（駅名を内包。同名遠方駅は #1, #2 ... に分かれる）

print("駅グループ数:", gdf["grp"].nunique(), " / 元の行数:", len(gdf))

駅グループ数: 9273  / 元の行数: 10534


In [8]:
# (grp, 運営会社, 年) に集約 → 運営会社×年 のワイド表（present / pax）
_rows = []
for y in YEARS:
    g = (gdf.groupby(["grp", "運営会社"])
            .agg(present=(f"データ有無コード{y}", lambda s: bool((s == 1).any())),
                 pax=(f"乗降客数{y}", "sum"))                  # 重複行は0なので二重計上なし
            .reset_index())
    g["year"] = y
    _rows.append(g)
op_year = pd.concat(_rows, ignore_index=True)

pres = (op_year.pivot_table(index=["grp", "運営会社"], columns="year",
                            values="present", aggfunc="first").fillna(False)[YEARS])
paxw = (op_year.pivot_table(index=["grp", "運営会社"], columns="year",
                            values="pax", aggfunc="first").fillna(0.0)[YEARS])
print("運営会社×群:", len(pres))

運営会社×群: 9781


### 指標① レベル時系列（inclusive・補間なし）

その年に実測のある全社を合算。欠測は0扱いせず、全社欠測の年は NaN（グラフは線を切る）。
社が抜けた年は寄与社数 `n_op` が下がるので、谷が「データ穴」か「実減」か判別できる。

In [9]:
# 各年の合算（在の社のみ）と、群の延べ社数・構成一定フラグ
present_long = op_year[op_year["present"]]

level = (present_long.groupby(["grp", "year"])
         .agg(pax=("pax", "sum"))
         .reset_index())
pax_wide = level.pivot(index="grp", columns="year", values="pax").reindex(columns=YEARS)
pax_wide.columns = [f"pax_{y}" for y in YEARS]   # 欠測年は自然に NaN

# 群の延べ運営会社数
n_op = pres.reset_index().groupby("grp")["運営会社"].nunique()

# level_complete: 構成が全データ年で一定か
#   = 群に現れる全社が、群がデータを持つ全年で在 であること
op_present_years = present_long.groupby(["grp", "運営会社"])["year"].nunique()
grp_data_years   = present_long.groupby("grp")["year"].nunique()
min_op_years     = op_present_years.groupby("grp").min()
level_complete   = (min_op_years == grp_data_years.reindex(min_op_years.index))

print("構成一定の群:", int(level_complete.sum()), "/", len(level_complete))

構成一定の群: 8434 / 8598


### 指標② パターン1（直近 2024/2023）

2023・2024の両方で実測があり、2023値>0 の社だけを basket に合算して比率を取る。

In [10]:
# パターン1: rate_yoy と異常値フラグ
m1 = pres[2023] & pres[2024] & (paxw[2023] > 0)
d1 = paxw.loc[m1, 2023].groupby(level="grp").sum()
n1 = paxw.loc[m1, 2024].groupby(level="grp").sum()
rate_yoy = (n1 / d1 - 1.0)
rate_yoy = rate_yoy[d1 > 0]
flag_yoy = rate_yoy.abs() > 0.30          # 事業者内の報告基準変更などを検出

print("パターン1 算出可能:", len(rate_yoy), "群  中央値: {:+.2f}%".format(rate_yoy.median()*100))

パターン1 算出可能: 7367 群  中央値: +2.14%


### 指標③ パターン2（コロナ前→後・事業者単位）

各社について「pre窓(2017-19)の最新実測」→「post窓(2023-24)の最新実測」を取り、
両方そろう社だけを basket に合算。参照年は社ごとに異なってよい（各社は自分基準で like-for-like）。

In [11]:
# パターン2: rate_covid と信頼性フラグ
def _latest(years_win):
    """各(grp,運営会社)について、窓内で present かつ pax>0 の最新年と値を返す"""
    yr  = pd.Series(np.nan, index=pres.index)
    val = pd.Series(0.0,    index=pres.index)
    for y in years_win:                       # 昇順に上書き → 最大年が残る
        ok  = pres[y] & (paxw[y] > 0)
        yr  = yr.mask(ok, y)
        val = val.mask(ok, paxw[y])
    return yr, val

pre_year,  pre_val  = _latest(PRE_WINDOW)
post_year, post_val = _latest(POST_WINDOW)

basket2 = pre_year.notna() & post_year.notna()           # pre/post 両方ある社のみ（pre/post一貫）
d2 = pre_val[basket2].groupby(level="grp").sum()
n2 = post_val[basket2].groupby(level="grp").sum()
rate_covid = (n2 / d2 - 1.0)
rate_covid = rate_covid[d2 > 0]

# 品質: 被覆率(basket社数/延べ社数) と pre年の最古
n_basket  = basket2.groupby(level="grp").sum()
cov_covid = n_basket / n_op.reindex(n_basket.index)
min_pre   = pre_year[basket2].groupby(level="grp").min()
flag_covid = (((cov_covid < 1.0) | (min_pre < 2019)).reindex(rate_covid.index, fill_value=True)
              | (rate_covid.abs() > 1.0))   # 極端値(|率|>100%)も要注意（小駅の分母ノイズ等。rate_yoyと設計統一）

print("パターン2 算出可能:", len(rate_covid), "群  中央値: {:+.2f}%".format(rate_covid.median()*100))

パターン2 算出可能: 7684 群  中央値: -8.40%


### メイン / 監査用テーブルの組み立て

In [12]:
# メイン station_group（1群1行）。今後 人口・地価などはこの右に追記していく
ident = gdf.groupby("grp").agg(駅名=("駅名", "first"), lon=("lon", "mean"), lat=("lat", "mean"))

station_group = ident.join(pax_wide)
station_group["n_op"]           = n_op.reindex(station_group.index)
station_group["level_complete"] = level_complete.reindex(station_group.index, fill_value=True)
station_group["rate_yoy"]       = (rate_yoy * 100).round(1).reindex(station_group.index)
station_group["flag_yoy"]       = flag_yoy.reindex(station_group.index, fill_value=False)
station_group["rate_covid"]     = (rate_covid * 100).round(1).reindex(station_group.index)
station_group["flag_covid"]     = flag_covid.reindex(station_group.index, fill_value=False)

for y in YEARS:                                   # NaN許容整数に
    station_group[f"pax_{y}"] = station_group[f"pax_{y}"].round().astype("Int64")
station_group = station_group.reset_index()

print("station_group:", station_group.shape)
station_group.head()

station_group: (9273, 24)


,grp,駅名,lon,lat,pax_2011,pax_2012,pax_2013,pax_2014,pax_2015,pax_2016,pax_2017,pax_2018,pax_2019,pax_2020,pax_2021,pax_2022,pax_2023,pax_2024,n_op,level_complete,rate_yoy,flag_yoy,rate_covid,flag_covid
0,JA広島病院前#0,JA広島病院前,132.32251,34.34405,3403,3403,3403,3403,3403,3403,1674,1548,1474,1153,1085,1178,1270,1474,1,True,16.1,False,0.0,False
1,JR三山木#0,JR三山木,135.78427,34.79899,1484,1586,1613,1676,1800,1866,1899,2004,2138,1626,1678,1762,1924,2052,1,True,6.7,False,-4.0,False
2,JR五位堂#0,JR五位堂,135.71831,34.52768,1521,1493,1448,1444,1484,1526,1567,1576,1514,1246,1314,1364,1436,1424,1,True,-0.8,False,-5.9,False
3,JR俊徳道#0,JR俊徳道,135.57249,34.65830,6471,6899,7347,7512,7802,7948,8147,8394,10402,8108,9494,11170,12180,13006,1,True,6.8,False,25.0,False
4,JR小倉#0,JR小倉,135.78634,34.88913,3817,3828,3871,3816,3844,3724,3752,3734,3700,2828,2944,3282,3632,3992,1,True,9.9,False,7.9,False


In [13]:
# 監査用 station_group_operator（1群×1社）。各年値・在/欠測・参照年を保持
_pax = paxw.where(pres, np.nan)                    # 欠測は NaN（実測の0と区別）
_pax.columns  = [f"pax_{y}" for y in YEARS]
_present = pres.copy()
_present.columns = [f"present_{y}" for y in YEARS]

station_group_operator = _pax.join(_present)
station_group_operator["pre_year"]  = pre_year
station_group_operator["post_year"] = post_year
for y in YEARS:
    station_group_operator[f"pax_{y}"] = station_group_operator[f"pax_{y}"].round().astype("Int64")
station_group_operator = station_group_operator.reset_index()

print("station_group_operator:", station_group_operator.shape)
station_group_operator.head()

station_group_operator: (9781, 32)


,grp,運営会社,pax_2011,pax_2012,pax_2013,pax_2014,pax_2015,pax_2016,pax_2017,pax_2018,pax_2019,pax_2020,pax_2021,pax_2022,pax_2023,pax_2024,present_2011,present_2012,present_2013,present_2014,present_2015,present_2016,present_2017,present_2018,present_2019,present_2020,present_2021,present_2022,present_2023,present_2024,pre_year,post_year
0,JA広島病院前#0,広島電鉄,3403,3403,3403,3403,3403,3403,1674,1548,1474,1153,1085,1178,1270,1474,True,True,True,True,True,True,True,True,True,True,True,True,True,True,2019.0,2024.0
1,JR三山木#0,西日本旅客鉄道,1484,1586,1613,1676,1800,1866,1899,2004,2138,1626,1678,1762,1924,2052,True,True,True,True,True,True,True,True,True,True,True,True,True,True,2019.0,2024.0
2,JR五位堂#0,西日本旅客鉄道,1521,1493,1448,1444,1484,1526,1567,1576,1514,1246,1314,1364,1436,1424,True,True,True,True,True,True,True,True,True,True,True,True,True,True,2019.0,2024.0
3,JR俊徳道#0,西日本旅客鉄道,6471,6899,7347,7512,7802,7948,8147,8394,10402,8108,9494,11170,12180,13006,True,True,True,True,True,True,True,True,True,True,True,True,True,True,2019.0,2024.0
4,JR小倉#0,西日本旅客鉄道,3817,3828,3871,3816,3844,3724,3752,3734,3700,2828,2944,3282,3632,3992,True,True,True,True,True,True,True,True,True,True,True,True,True,True,2019.0,2024.0


## （３）都道府県・駅名表記・検索名表記

国土数値情報「行政区域(N03)」の都道府県ポリゴンを群の代表点に空間結合（`gpd.sjoin`）して `都道府県` を付与し、表示用 `label` と検索用 `search_label` を生成する。

- **label（表示用）**: 単独駅名→`駅名` ／ 同名複数でも「複数運営会社」「同一社が別地域に衝突」は `駅名`（素）／ 単独運営会社で衝突なしは `駅名（運営会社）`
- **search_label（検索用）**: `label` に `都道府県` を付加（単一括弧）。例: 尼崎（阪神電気鉄道・兵庫県）／ 二月田（鹿児島県）／ 上道（鳥取県）
- `grp` は結合キーとして温存。`都道府県` も単独列で保持。

In [14]:
# 行政区域(N03)の都道府県ポリゴンを代表点に空間結合して 都道府県 を付与
N03_ZIP = BASE_DIR / "data" / "行政区域データ" / "N03-20260101_GML.zip"
# 都道府県名でまとまった _prefecture 版を使用。CRS は EPSG:6668 で S12 と一致（再投影不要）
pref = gpd.read_file(f"zip://{N03_ZIP}!N03-20260101_prefecture.shp", columns=["N03_001"])
pref = pref.rename(columns={"N03_001": "都道府県"})

grp_pt = gpd.GeoDataFrame(
    station_group[["grp"]].copy(),
    geometry=gpd.points_from_xy(station_group["lon"], station_group["lat"]),
    crs="EPSG:6668",
)
# 点が含まれる都道府県を判定。境界外（海上の埋立地・座標誤差）は最近傍で補完
hit = gpd.sjoin(grp_pt, pref, how="left", predicate="within").drop_duplicates("grp")[["grp", "都道府県"]]
miss = hit["都道府県"].isna()
if miss.any():
    near = (gpd.sjoin_nearest(grp_pt[grp_pt["grp"].isin(hit.loc[miss, "grp"])].to_crs(3857), pref.to_crs(3857), how="left")
            .drop_duplicates("grp").set_index("grp")["都道府県"])
    hit.loc[miss, "都道府県"] = hit.loc[miss, "grp"].map(near)

station_group = station_group.merge(hit, on="grp", how="left")
print("都道府県 付与:", int(station_group["都道府県"].notna().sum()), "/", len(station_group),
      " 未割当:", int(station_group["都道府県"].isna().sum()))
print(station_group["都道府県"].value_counts().head())

都道府県 付与: 9273 / 9273  未割当: 0
都道府県
東京都    654
北海道    558
大阪府    480
愛知県    477
兵庫県    365
Name: count, dtype: int64


In [15]:
# 表示用 label と 検索用 search_label を生成
ops_by_grp = (station_group_operator.groupby("grp")["運営会社"]
              .apply(lambda s: sorted(s.unique().tolist())).to_dict())
grps_by_name = station_group.groupby("駅名")["grp"].apply(list).to_dict()

def make_label(nm, grp):
    grps = grps_by_name[nm]
    if len(grps) == 1:                               # A: 単独駅名
        return nm
    ops = ops_by_grp.get(grp, [])
    if len(ops) != 1:                                # C: 複数運営会社 → 素
        return nm
    op = ops[0]
    for g in grps:                                   # D: 同一単独社が別地域に衝突 → 素
        if g != grp and ops_by_grp.get(g, []) == [op]:
            return nm
    return f"{nm}（{op}）"                             # B: 単独社・衝突なし → 社付き

station_group["label"] = [make_label(r["駅名"], r["grp"]) for _, r in station_group.iterrows()]

def make_search_label(label, pref):
    if pd.isna(pref):
        return label
    if label.endswith(f"（{pref}）"):                  # 運営会社名が都道府県名と同一（例: 都営=東京都）
        return label
    if label.endswith("）"):                          # 駅名（運営会社） → 駅名（運営会社・都道府県）
        return label[:-1] + "・" + pref + "）"
    return f"{label}（{pref}）"                         # 駅名 → 駅名（都道府県）

station_group["search_label"] = [make_search_label(l, p)
                                 for l, p in zip(station_group["label"], station_group["都道府県"])]

# 列順を整える（識別子を前方へ）
front = ["grp", "駅名", "label", "search_label", "都道府県", "lon", "lat"]
station_group = station_group[front + [c for c in station_group.columns if c not in front]]

# 検索ラベルの一意性チェック（将来データで同名・同県・素ラベルが衝突したら検知）
print("search_label 重複:", int(station_group["search_label"].duplicated().sum()), "（0 が正常）")
station_group.head()

search_label 重複: 0 （0 が正常）


,grp,駅名,label,search_label,都道府県,lon,lat,pax_2011,pax_2012,pax_2013,pax_2014,pax_2015,pax_2016,pax_2017,pax_2018,pax_2019,pax_2020,pax_2021,pax_2022,pax_2023,pax_2024,n_op,level_complete,rate_yoy,flag_yoy,rate_covid,flag_covid
0,JA広島病院前#0,JA広島病院前,JA広島病院前,JA広島病院前（広島県）,広島県,132.32251,34.34405,3403,3403,3403,3403,3403,3403,1674,1548,1474,1153,1085,1178,1270,1474,1,True,16.1,False,0.0,False
1,JR三山木#0,JR三山木,JR三山木,JR三山木（京都府）,京都府,135.78427,34.79899,1484,1586,1613,1676,1800,1866,1899,2004,2138,1626,1678,1762,1924,2052,1,True,6.7,False,-4.0,False
2,JR五位堂#0,JR五位堂,JR五位堂,JR五位堂（奈良県）,奈良県,135.71831,34.52768,1521,1493,1448,1444,1484,1526,1567,1576,1514,1246,1314,1364,1436,1424,1,True,-0.8,False,-5.9,False
3,JR俊徳道#0,JR俊徳道,JR俊徳道,JR俊徳道（大阪府）,大阪府,135.57249,34.65830,6471,6899,7347,7512,7802,7948,8147,8394,10402,8108,9494,11170,12180,13006,1,True,6.8,False,25.0,False
4,JR小倉#0,JR小倉,JR小倉,JR小倉（京都府）,京都府,135.78634,34.88913,3817,3828,3871,3816,3844,3724,3752,3734,3700,2828,2944,3282,3632,3992,1,True,9.9,False,7.9,False


In [16]:
# 確認: ラベル生成の代表例（A:単独 / B:社付き / C:複数社→素 / D:同一社衝突→素）
_chk = ["二月田#0", "尼崎#0", "尼崎#1", "市役所前#0", "上道#0", "上道#1",
        "大久保#2", "大久保#3", "三田#0", "三田#1"]
station_group[station_group["grp"].isin(_chk)][["grp", "label", "search_label", "都道府県"]]

,grp,label,search_label,都道府県
389,三田#0,三田,三田（兵庫県）,兵庫県
390,三田#1,三田（東京都）,三田（東京都）,東京都
572,上道#0,上道,上道（鳥取県）,鳥取県
573,上道#1,上道,上道（岡山県）,岡山県
982,二月田#0,二月田,二月田（鹿児島県）,鹿児島県
2701,大久保#2,大久保,大久保（秋田県）,秋田県
2702,大久保#3,大久保,大久保（東京都）,東京都
3512,尼崎#0,尼崎（阪神電気鉄道）,尼崎（阪神電気鉄道・兵庫県）,兵庫県
3513,尼崎#1,尼崎（西日本旅客鉄道）,尼崎（西日本旅客鉄道・兵庫県）,兵庫県
3830,市役所前#0,市役所前（伊予鉄道）,市役所前（伊予鉄道・愛媛県）,愛媛県


In [17]:
# 確認: 代表的な駅（東京・新宿・渋谷・横浜・小諸）
_check = ["東京#0", "新宿#0", "渋谷#0", "横浜#0", "小諸#0"]
_cols = ["grp", "駅名", "n_op", "level_complete",
         "pax_2019", "pax_2024", "rate_yoy", "flag_yoy", "rate_covid", "flag_covid"]
station_group[station_group["grp"].isin(_check)][_cols]

,grp,駅名,n_op,level_complete,pax_2019,pax_2024,rate_yoy,flag_yoy,rate_covid,flag_covid
3481,小諸#0,小諸,2,False,6518,5647,119.0,True,58.0,True
4376,新宿#0,新宿,5,True,3265570,2713386,-9.1,False,-16.9,True
4933,東京#0,東京,3,False,1339555,1262604,7.3,False,-5.7,False
5524,横浜#0,横浜,6,True,2101709,1936287,-1.3,False,-7.9,True
6039,渋谷#0,渋谷,4,True,2704384,2897703,36.1,True,7.1,False


## （４）駅半径内人口（メッシュ面積按分・複数年）

`docs/population_mesh.md` §8 に基づき、駅代表点の半径 500m/1km/2km/5km/10km/20km で**面積按分**集計する。
**2020・2015年は250mメッシュ、2010・2005・2000・1995年は500mメッシュ**（2010以前の250mは全国非対応のため、全国網羅の500mを採用）。列名で年を区別（`pop_{year}_*`）。
- メッシュ幾何はコードから生成（境界shp不要）、正積図法で面積按分、内部/境界分割＋KDTree。
- `load_mesh_pop` は **解像度(9桁=500m / 10桁=250m)を自動判定**し、判定マージン `H`（投影後セル半対角の上限）も自動算出。
- 人口コードは各年 `cat01=0010` 共通。秘匿/合算フラグは2020/2015のみ（2010 500mは列がない→`hidden_ratio=NaN`）。
- 出力: 各年 `pop_{year}_500m … pop_{year}_20km`（整数）＋ `pop_{year}_500m_hidden_ratio`。

In [18]:
# 半径内人口の共通処理（複数年・複数解像度で再利用）— docs/population_mesh.md §8 準拠
import io, glob, time
import numpy as np
from pyproj import Transformer
from scipy.spatial import cKDTree
import shapely

RADII = [500, 1000, 2000, 5000, 10000, 20000]
RNAME = {500:"500m", 1000:"1km", 2000:"2km", 5000:"5km", 10000:"10km", 20000:"20km"}
QS = 32          # バッファ多角形分割（円面積99.96%再現）
AEA = "+proj=aea +lat_1=29.5 +lat_2=45.5 +lat_0=35 +lon_0=135 +x_0=0 +y_0=0 +ellps=GRS80 +units=m +no_defs"
tf_aea = Transformer.from_crs("EPSG:6668", AEA, always_xy=True)   # CRS.md の正積図法

def _build_geom(area):
    """メッシュコード列(str)→ 矩形ポリゴン・面積・重心・KDTree・判定マージンH。解像度はコード桁数で自動判定(10桁=250m/9桁=500m)。"""
    s = area.astype(str); L = int(s.str.len().mode()[0])
    a=s.str[0:2].astype(int); b=s.str[2:4].astype(int); c=s.str[4].astype(int); d=s.str[5].astype(int)
    e=s.str[6].astype(int); f=s.str[7].astype(int); g=s.str[8].astype(int)
    lat = a*2/3 + c*(2/3)/8 + e*(2/3)/80 + ((g-1)//2)*(1/240)
    lon = (b+100) + d/8 + f/80 + ((g-1)%2)*(1/160)
    if L == 10:                                  # 250m: 5次象限を追加
        hc = s.str[9].astype(int)
        lat = lat + ((hc-1)//2)*(1/480); lon = lon + ((hc-1)%2)*(1/320); dlat, dlon = 1/480, 1/320
    else:                                        # 9桁 = 500m
        dlat, dlon = 1/240, 1/160
    lat_s = lat.to_numpy(); lon_w = lon.to_numpy(); lat_n = lat_s+dlat; lon_e = lon_w+dlon
    xw,ys=tf_aea.transform(lon_w,lat_s); xe,ys2=tf_aea.transform(lon_e,lat_s)
    xe2,yn=tf_aea.transform(lon_e,lat_n); xw2,yn2=tf_aea.transform(lon_w,lat_n)
    rings=np.stack([np.c_[xw,ys],np.c_[xe,ys2],np.c_[xe2,yn],np.c_[xw2,yn2],np.c_[xw,ys]],axis=1)
    mcx=rings[:,:4,0].mean(1); mcy=rings[:,:4,1].mean(1)
    xx=rings[:,:,0]; yy=rings[:,:,1]
    area_a=0.5*np.abs(np.sum(xx[:,:-1]*yy[:,1:]-xx[:,1:]*yy[:,:-1],axis=1))   # shoelace=真の地表面積
    halfdiag=np.sqrt(((rings[:,:4,0]-mcx[:,None])**2+(rings[:,:4,1]-mcy[:,None])**2).max(1))
    return dict(rings=rings, mcx=mcx, mcy=mcy, area=area_a, tree=cKDTree(np.c_[mcx,mcy]),
                H=float(halfdiag.max())*1.02, res_m=(250 if L==10 else 500))

def load_mesh_pop(mesh_dir, pop_cat01=10):
    """国勢調査メッシュCSV群から人口総数を読み、geom + pop + 秘匿フラグを構築。"""
    recs = []; has_secrecy = False
    for fp in sorted(glob.glob(str(mesh_dir / "mesh*.csv"))):
        lines = Path(fp).read_text(encoding="utf-8").splitlines()
        vi = next(i for i, l in enumerate(lines) if l == '"VALUE"')
        tbl = pd.read_csv(io.StringIO(chr(10).join(lines[vi+1:])))
        sub = tbl[pd.to_numeric(tbl["cat01_code"], errors="coerce") == pop_cat01]
        rec = pd.DataFrame({"area": sub["area_code"].astype(str),
                            "pop": pd.to_numeric(sub["value"], errors="coerce").fillna(0.0)})
        if "cat02_code" in tbl.columns:
            has_secrecy = True
            nm = sub.iloc[:, tbl.columns.get_loc("cat02_code") + 1].astype(str)
            rec["hidden"] = nm.isin(["合算", "秘匿"]).to_numpy()
        else:
            rec["hidden"] = False
        recs.append(rec)
    m = pd.concat(recs, ignore_index=True)
    M = _build_geom(m["area"])
    M.update(pop=m["pop"].to_numpy(), hidden=m["hidden"].to_numpy(),
             has_secrecy=has_secrecy, total=int(m["pop"].sum()), ncell=len(m))
    return M

def aggregate_radii(M, sx, sy):
    """1駅について各半径の人口と500m秘匿割合（docs §8.4）。Hはメッシュ依存。"""
    H = M["H"]; out={R:0.0 for R in RADII}; hid500=0.0
    cand=np.asarray(M["tree"].query_ball_point([sx,sy], RADII[-1]+H), dtype=int)
    if cand.size==0: return out, 0.0
    dist=np.hypot(M["mcx"][cand]-sx, M["mcy"][cand]-sy)
    for R in RADII:
        ins=dist<=R-H; band=(dist>R-H)&(dist<=R+H)
        pin=M["pop"][cand][ins]; tot=pin.sum(); hidp=pin[M["hidden"][cand][ins]].sum()
        if band.any():
            bi=cand[band]
            w=shapely.area(shapely.intersection(shapely.polygons(M["rings"][bi]),
                                                shapely.Point(sx,sy).buffer(R,quad_segs=QS)))/M["area"][bi]
            bp=M["pop"][bi]*w; tot+=bp.sum(); hidp+=bp[M["hidden"][bi]].sum()
        out[R]=tot
        if R==500 and tot>0: hid500=hidp/tot
    return out, hid500

def aggregate_radii_multi(M, pops, sx, sy):
    """複数年の人口配列(pops: {year: array})を一括集計。境界weightは年共通なので1駅×半径で1回だけ計算。"""
    H = M["H"]; out={y:{R:0.0 for R in RADII} for y in pops}
    cand=np.asarray(M["tree"].query_ball_point([sx,sy], RADII[-1]+H), dtype=int)
    if cand.size==0: return out
    dist=np.hypot(M["mcx"][cand]-sx, M["mcy"][cand]-sy)
    for R in RADII:
        ins=dist<=R-H; band=(dist>R-H)&(dist<=R+H)
        bi = cand[band] if band.any() else None
        if bi is not None:
            w=shapely.area(shapely.intersection(shapely.polygons(M["rings"][bi]),
                                                shapely.Point(sx,sy).buffer(R,quad_segs=QS)))/M["area"][bi]
        for y, parr in pops.items():
            tot = parr[cand][ins].sum()
            if bi is not None: tot += (parr[bi]*w).sum()
            out[y][R]=tot
    return out

In [19]:
# 年ごとにメッシュ人口を算出 → pop_{year}_<半径> 列を付与（2020/2015=250m, 2010/2005/2000/1995=500m）
MESH_YEARS = {
    2020: BASE_DIR / "data" / "国勢調査_人口及び世帯_2020_mesh250",
    2015: BASE_DIR / "data" / "国勢調査_人口及び世帯_2015_mesh250",
    2010: BASE_DIR / "data" / "国勢調査_人口及び世帯_2010_mesh500",
    2005: BASE_DIR / "data" / "国勢調査_人口及び世帯_2005_mesh500",
    2000: BASE_DIR / "data" / "国勢調査_人口及び世帯_2000_mesh500",
    1995: BASE_DIR / "data" / "国勢調査_人口及び世帯_1995_mesh500",
}
_sx, _sy = tf_aea.transform(station_group["lon"].to_numpy(), station_group["lat"].to_numpy())
nST = len(station_group)
for year, mdir in MESH_YEARS.items():
    M = load_mesh_pop(mdir); hs = M["has_secrecy"]
    print(f"[{year}] {M['res_m']}mメッシュ {M['ncell']:,}セル 全国人口計 {M['total']:,}")
    pop_out = {R: np.zeros(nST) for R in RADII}; hid500 = np.zeros(nST)
    t0 = time.time()
    for j in range(nST):
        o, hr = aggregate_radii(M, _sx[j], _sy[j])
        for R in RADII: pop_out[R][j] = o[R]
        hid500[j] = hr
        if (j+1) % 3000 == 0: print(f"  [{year}] {j+1}/{nST} ({time.time()-t0:.0f}s)")
    arr = np.maximum.accumulate(np.column_stack([pop_out[R] for R in RADII]), axis=1)   # 単調化（float誤差対策）
    for k, R in enumerate(RADII):
        station_group[f"pop_{year}_{RNAME[R]}"] = pd.Series(np.rint(arr[:, k]), index=station_group.index).astype("Int64")
    station_group[f"pop_{year}_500m_hidden_ratio"] = np.round(hid500, 3) if hs else np.nan   # 2010は秘匿フラグなし
    print(f"  [{year}] 集計完了 {time.time()-t0:.0f}s  (秘匿フラグ: {'有' if hs else '無'})")
    del M

[2020] 250mメッシュ 1,163,757セル 全国人口計 126,146,099


  [2020] 3000/9273 (31s)


  [2020] 6000/9273 (64s)


  [2020] 9000/9273 (95s)


  [2020] 集計完了 98s  (秘匿フラグ: 有)


...

[1995] 500mメッシュ 312,515セル 全国人口計 125,570,246


  [1995] 3000/9273 (17s)


  [1995] 6000/9273 (34s)


  [1995] 9000/9273 (51s)


  [1995] 集計完了 52s  (秘匿フラグ: 無)


In [20]:
# 検証: 各年の単調性 ＋ 代表駅の年次比較（2020/2015/2010）
for year in MESH_YEARS:
    _o = [f"pop_{year}_{RNAME[R]}" for R in RADII]
    _mono = (station_group[_o].to_numpy()[:, 1:] >= station_group[_o].to_numpy()[:, :-1]).all()
    print(f"{year} 単調性 OK:", bool(_mono))
_yrs = sorted(MESH_YEARS, reverse=True)
_cols = [f"pop_{y}_{r}" for r in ["1km","5km","20km"] for y in _yrs]
station_group[station_group["grp"].isin(["東京#0","新宿#0","横浜#0","二月田#0"])][["grp","駅名"]+_cols]

2020 単調性 OK: True
2015 単調性 OK: True
2010 単調性 OK: True
2005 単調性 OK: True
2000 単調性 OK: True
1995 単調性 OK: True


,grp,駅名,pop_2020_1km,pop_2015_1km,pop_2010_1km,pop_2005_1km,pop_2000_1km,pop_1995_1km,pop_2020_5km,pop_2015_5km,pop_2010_5km,pop_2005_5km,pop_2000_5km,pop_1995_5km,pop_2020_20km,pop_2015_20km,pop_2010_20km,pop_2005_20km,pop_2000_20km,pop_1995_20km
982,二月田#0,二月田,4292,4175,4061,4081,4125,4201,25794,27038,27977,28913,29767,30442,68994,75470,81103,86390,90473,94034
4376,新宿#0,新宿,25571,24831,26067,23603,22010,21679,1380338,1307184,1244670,1189099,1147803,1119800,15461814,14786688,14298231,13587834,12992867,12635329
4933,東京#0,東京,4248,4126,5510,4412,3252,3239,1186130,1071653,957108,858654,750608,717317,13998943,13367580,12924701,12272199,11728443,11463811
5524,横浜#0,横浜,43471,41272,39004,31951,25579,22375,820477,791114,774928,743203,703679,678883,8561345,8342470,8181134,7879518,7546950,7307187


## （５）人口増減率（各半径・指定年ペア）

§（４）の年次人口から、各半径の**人口増減率**を算出する（`docs/population_mesh.md` §8.7）。
- **2020基準**（各年との比較）: 2020/2015, 2020/2010, 2020/2005, 2020/2000, 2020/1995
- **直近**（前回国勢調査との比較）: 2015/2010, 2010/2005, 2005/2000, 2000/1995
- 列名 **`pop_gr_{新しい年}_{古い年}_{半径}`**（単位%・小数1桁、分母0/欠損はNaN）。9ペア×6半径=**54列**。
- ⚠️ 250m/500m境界を跨ぐペア（2020・2015 × 2010以前）は **500m/1km半径にメッシュ段差(~3.6%/1%)** を含む（2km以上はクリーン。§6.3）。同一解像度ペア（2020/2015、および 2010/2005/2000/1995）は完全整合。

In [21]:
# 人口増減率: 指定年ペア × 各半径 → pop_gr_{新}_{旧}_{半径}（%）
POP_PAIRS = [
    (2020, 2015), (2020, 2010), (2020, 2005), (2020, 2000), (2020, 1995),  # 2020基準（各年との比較）
    (2015, 2010), (2010, 2005), (2005, 2000), (2000, 1995),                # 直近（前回国勢調査との比較）
]
for newer, older in POP_PAIRS:
    for R in RADII:
        new = station_group[f"pop_{newer}_{RNAME[R]}"].to_numpy(dtype="float64", na_value=np.nan)
        old = station_group[f"pop_{older}_{RNAME[R]}"].to_numpy(dtype="float64", na_value=np.nan)
        with np.errstate(divide="ignore", invalid="ignore"):
            gr = np.where(old > 0, (new / old - 1.0) * 100.0, np.nan)   # 分母0/欠損は NaN
        station_group[f"pop_gr_{newer}_{older}_{RNAME[R]}"] = np.round(gr, 1)

_grcols = [c for c in station_group.columns if c.startswith("pop_gr_")]
print(f"人口増減率 列数: {len(_grcols)}（{len(POP_PAIRS)}ペア × {len(RADII)}半径）  NaNを含む列: {sum(station_group[c].isna().any() for c in _grcols)}")
# サニティ: 東京(都心回帰で増)・二月田(地方減) の5km
station_group[station_group["grp"].isin(["東京#0","二月田#0"])][
    ["駅名","pop_gr_2020_1995_5km","pop_gr_2020_2015_5km","pop_gr_2015_2010_5km","pop_gr_2000_1995_5km"]]

人口増減率 列数: 54（9ペア × 6半径）  NaNを含む列: 34


,駅名,pop_gr_2020_1995_5km,pop_gr_2020_2015_5km,pop_gr_2015_2010_5km,pop_gr_2000_1995_5km
982,二月田,-15.3,-4.6,-3.4,-2.2
4933,東京,65.4,10.7,12.0,4.6


## （６）将来推計人口（R6・250mメッシュ, 2020–2070）

国土数値情報「将来推計人口250mメッシュ（R6国政局推計）」を、§（４）と同じ面積按分で駅半径内に集計（`docs/population_mesh.md` §10）。
- **真値フィールド `PTN_<year>`**（秘匿なし＝合算前の各セル値）を使用（合算による所在ズレを回避）。
- 都道府県別の**入れ子zip**を展開し、跨ぎメッシュは `MESH_ID` で合算（重複解消）。
- メッシュコード(10桁=250m)を §8 デコーダで矩形化 → 既存パイプライン（内部/境界分割・H自動）を再利用。
- 出力 **`pop_pred_2024_<年>_<半径>`**（R6＝2024年作成。年＝2020基準実績＋2025–2070推計。11年×6半径＝66列）。`pop_pred_2024_2020` は実績 `pop_2020` とほぼ一致（検証）。

In [22]:
# 将来推計人口 R6（250m）の読込: 入れ子zip→県別shp→MESH_ID + PTN_<year>
import zipfile, tempfile
R6_OUTER = BASE_DIR / "data" / "250mメッシュ別将来推計人口データ（R6国政局推計）" / "250m_mesh_2024_SHP.zip"
PROJ_YEARS = [2020, 2025, 2030, 2035, 2040, 2045, 2050, 2055, 2060, 2065, 2070]  # 2020=基準実績, 2025-2070=推計
_ptn = [f"PTN_{y}" for y in PROJ_YEARS]
_parts = []
with zipfile.ZipFile(R6_OUTER) as z:
    for n in sorted(x for x in z.namelist() if x.endswith("_SHP.zip")):   # 47都道府県
        with tempfile.TemporaryDirectory() as td:
            z.extract(n, td)
            with zipfile.ZipFile(Path(td) / n) as z2:
                z2.extractall(td)
            shp = next(Path(td).rglob("*.shp"))
            _parts.append(pd.DataFrame(gpd.read_file(shp, columns=["MESH_ID"] + _ptn, ignore_geometry=True)))
r6 = pd.concat(_parts, ignore_index=True)
# 都道府県跨ぎの分割メッシュ(同一MESH_IDが複数県に分割計上)を合算して統合
r6 = r6.groupby("MESH_ID", as_index=False)[_ptn].sum()
print(f"R6 250m: {len(r6):,}セル  PTN_2020={int(r6['PTN_2020'].sum()):,}（実績2020=126,146,099と僅差）  PTN_2070={int(r6['PTN_2070'].sum()):,}")

R6 250m: 1,162,630セル  PTN_2020=126,146,098（実績2020=126,146,099と僅差）  PTN_2070=86,996,003


In [23]:
# geometry を1回構築し、全推計年を一括集計（境界weightは年共通→高速）→ pop_pred_2024_<年>_<半径>
Mr6 = _build_geom(r6["MESH_ID"])
_pops = {y: r6[f"PTN_{y}"].fillna(0).to_numpy(dtype="float64") for y in PROJ_YEARS}
_sx, _sy = tf_aea.transform(station_group["lon"].to_numpy(), station_group["lat"].to_numpy())
nST = len(station_group)
_acc = {y: {R: np.zeros(nST) for R in RADII} for y in PROJ_YEARS}
_t0 = time.time()
for j in range(nST):
    o = aggregate_radii_multi(Mr6, _pops, _sx[j], _sy[j])
    for y in PROJ_YEARS:
        for R in RADII: _acc[y][R][j] = o[y][R]
    if (j+1) % 3000 == 0: print(f"  {j+1}/{nST} ({time.time()-_t0:.0f}s)")
for y in PROJ_YEARS:
    arr = np.maximum.accumulate(np.column_stack([_acc[y][R] for R in RADII]), axis=1)   # 単調化
    for k, R in enumerate(RADII):
        station_group[f"pop_pred_2024_{y}_{RNAME[R]}"] = pd.Series(np.rint(arr[:, k]), index=station_group.index).astype("Int64")
print(f"R6集計完了 {time.time()-_t0:.0f}s  追加列: {len([c for c in station_group.columns if c.startswith('pop_pred_2024_')])}")

  3000/9273 (34s)


  6000/9273 (68s)


  9000/9273 (101s)


R6集計完了 104s  追加列: 66


In [24]:
# 検証: 基準年(R6 2020)≒実績2020 / 将来の推移、各年の単調性
_bad = [y for y in PROJ_YEARS if not (station_group[[f"pop_pred_2024_{y}_{RNAME[R]}" for R in RADII]].to_numpy()[:,1:] >= station_group[[f"pop_pred_2024_{y}_{RNAME[R]}" for R in RADII]].to_numpy()[:,:-1]).all()]
print("単調性 違反年:", _bad if _bad else "なし(全年OK)")
station_group[station_group["grp"].isin(["東京#0","新宿#0","二月田#0"])][
    ["駅名","pop_2020_5km","pop_pred_2024_2020_5km","pop_pred_2024_2050_5km","pop_pred_2024_2070_5km"]]

単調性 違反年: なし(全年OK)


,駅名,pop_2020_5km,pop_pred_2024_2020_5km,pop_pred_2024_2050_5km,pop_pred_2024_2070_5km
982,二月田,25794,25771,16635,11722
4376,新宿,1380338,1379926,1480708,1404340
4933,東京,1186130,1182055,1379704,1342849


## （７）将来推計人口（H30・500mメッシュ, 2015–2050）

国土数値情報「将来推計人口500mメッシュ（H30国政局推計）」を §（６）と同じ面積按分で集計（`docs/population_mesh.md` §10）。
- 2015国勢調査ベース・2050まで。**真値 `PTN_<year>`** 使用。CRSは4612(JGD2000)だがメッシュコードから矩形生成（EPSG:6668扱い・差<1m）。
- 都道府県別zip（shp直下・入れ子なし）。メッシュ重複なし（groupbyは安全網）。
- 出力 **`pop_pred_2018_<年>_<半径>`**（H30＝2018年作成。2015基準＋2020–2050。8年×6半径＝48列）。
- 注: `pop_pred_2018_2020` はH30の2020「予測」（2018時点）であり、実績 `pop_2020` やR6 `pop_pred_2024_2020` とは別物（予測誤差の確認に有用）。

In [25]:
# 将来推計人口 H30（500m）の読込: 都道府県別zip(shp直下)→ MESH_ID + PTN_<year>
import zipfile
H30_DIR = BASE_DIR / "data" / "500mメッシュ別将来推計人口データ（H30国政局推計）"
PROJ_YEARS_H30 = [2015, 2020, 2025, 2030, 2035, 2040, 2045, 2050]  # 2015=基準実績, 2020-2050=推計
_ptn_h30 = [f"PTN_{y}" for y in PROJ_YEARS_H30]
_parts = []
for zp in sorted(H30_DIR.glob("500m_mesh_suikei_2018_shape_*.zip")):   # 47都道府県
    with zipfile.ZipFile(zp) as z:
        _shp = next(n for n in z.namelist() if n.endswith(".shp"))
    _parts.append(pd.DataFrame(gpd.read_file(f"zip://{zp}!{_shp}", columns=["MESH_ID"] + _ptn_h30, ignore_geometry=True)))
h30 = pd.concat(_parts, ignore_index=True)
h30["MESH_ID"] = h30["MESH_ID"].astype(str)
h30 = h30.groupby("MESH_ID", as_index=False)[_ptn_h30].sum()   # 念のため重複統合
print(f"H30 500m: {len(h30):,}セル  PTN_2015={int(h30['PTN_2015'].sum()):,}（実績2015=127,094,745と僅差）  PTN_2050={int(h30['PTN_2050'].sum()):,}")

H30 500m: 471,024セル  PTN_2015=127,094,385（実績2015=127,094,745と僅差）  PTN_2050=101,922,983


In [26]:
# geometry を1回構築し全推計年を一括集計 → pop_pred_2018_<年>_<半径>
Mh30 = _build_geom(h30["MESH_ID"])
_pops_h30 = {y: h30[f"PTN_{y}"].fillna(0).to_numpy(dtype="float64") for y in PROJ_YEARS_H30}
_sxh, _syh = tf_aea.transform(station_group["lon"].to_numpy(), station_group["lat"].to_numpy())
nST = len(station_group)
_acch = {y: {R: np.zeros(nST) for R in RADII} for y in PROJ_YEARS_H30}
_t0 = time.time()
for j in range(nST):
    o = aggregate_radii_multi(Mh30, _pops_h30, _sxh[j], _syh[j])
    for y in PROJ_YEARS_H30:
        for R in RADII: _acch[y][R][j] = o[y][R]
    if (j+1) % 3000 == 0: print(f"  {j+1}/{nST} ({time.time()-_t0:.0f}s)")
for y in PROJ_YEARS_H30:
    arr = np.maximum.accumulate(np.column_stack([_acch[y][R] for R in RADII]), axis=1)   # 単調化
    for k, R in enumerate(RADII):
        station_group[f"pop_pred_2018_{y}_{RNAME[R]}"] = pd.Series(np.rint(arr[:, k]), index=station_group.index).astype("Int64")
print(f"H30集計完了 {time.time()-_t0:.0f}s  追加列: {len([c for c in station_group.columns if c.startswith('pop_pred_2018_')])}")

  3000/9273 (22s)


  6000/9273 (45s)


  9000/9273 (66s)


H30集計完了 68s  追加列: 48


In [27]:
# 検証: 基準年(H30 2015)≒実績2015 / H30の2020予測 vs 実績・R6 / 単調性
_bad = [y for y in PROJ_YEARS_H30 if not (station_group[[f"pop_pred_2018_{y}_{RNAME[R]}" for R in RADII]].to_numpy()[:,1:] >= station_group[[f"pop_pred_2018_{y}_{RNAME[R]}" for R in RADII]].to_numpy()[:,:-1]).all()]
print("H30 単調性 違反年:", _bad if _bad else "なし(全年OK)")
station_group[station_group["grp"].isin(["東京#0","二月田#0"])][
    ["駅名","pop_2015_5km","pop_pred_2018_2015_5km","pop_2020_5km","pop_pred_2018_2020_5km","pop_pred_2024_2020_5km","pop_pred_2018_2050_5km","pop_pred_2024_2050_5km"]]

H30 単調性 違反年: なし(全年OK)


,駅名,pop_2015_5km,pop_pred_2018_2015_5km,pop_2020_5km,pop_pred_2018_2020_5km,pop_pred_2024_2020_5km,pop_pred_2018_2050_5km,pop_pred_2024_2050_5km
982,二月田,27038,27095,25794,25645,25771,16213,16635
4933,東京,1071653,1084871,1186130,1157019,1182055,1306333,1379704


## （８）将来人口増減率（実績2020 → R6推計各年）

「最新の確定実績（2020国勢調査）」を起点に、R6将来推計（2024）の各年への**人口増減率**を算出（`docs/population_mesh.md` §8.8）。
- 分母＝**実績 `pop_2020_{半径}`**（公式・最も正確な2020実績）。分子＝**`pop_pred_2024_{年}_{半径}`**（R6推計, 2025–2070）。
- 列名 **`pop_gr_pred_2024_{年}_{半径}`**（%・小数1桁、分母0/欠損はNaN）。10年×6半径＝**60列**。
- 起点は2020実績で固定。R6の2020基準(`pop_pred_2024_2020`)との出典差は~0.3%（許容）。両者250mでメッシュ段差なし。
- `pred_2024` で実績増減率 `pop_gr_{年}_{年}` と区別。

In [28]:
# 将来人口増減率: 実績2020(国勢調査) を起点に R6推計各年 → pop_gr_pred_2024_{年}_{半径}（%）
PRED_GR_YEARS = [2025, 2030, 2035, 2040, 2045, 2050, 2055, 2060, 2065, 2070]  # R6推計年（2020基準実績は分母なので除く）
_fgcols = {}
for year in PRED_GR_YEARS:
    for R in RADII:
        new = station_group[f"pop_pred_2024_{year}_{RNAME[R]}"].to_numpy(dtype="float64", na_value=np.nan)
        old = station_group[f"pop_2020_{RNAME[R]}"].to_numpy(dtype="float64", na_value=np.nan)   # 実績2020が分母
        with np.errstate(divide="ignore", invalid="ignore"):
            _fgcols[f"pop_gr_pred_2024_{year}_{RNAME[R]}"] = np.round(np.where(old > 0, (new / old - 1.0) * 100.0, np.nan), 1)
# 断片化回避のため一括結合
station_group = pd.concat([station_group, pd.DataFrame(_fgcols, index=station_group.index)], axis=1)
_fg = [c for c in station_group.columns if c.startswith("pop_gr_pred_2024_")]
print(f"将来人口増減率 列数: {len(_fg)}（{len(PRED_GR_YEARS)}年 × {len(RADII)}半径）  NaNを含む列: {sum(station_group[c].isna().any() for c in _fg)}")
# サニティ: 東京(増)・二月田(減) の5km
station_group[station_group["grp"].isin(["東京#0","二月田#0"])][
    ["駅名","pop_gr_pred_2024_2030_5km","pop_gr_pred_2024_2050_5km","pop_gr_pred_2024_2070_5km"]]

将来人口増減率 列数: 60（10年 × 6半径）  NaNを含む列: 30


,駅名,pop_gr_pred_2024_2030_5km,pop_gr_pred_2024_2050_5km,pop_gr_pred_2024_2070_5km
982,二月田,-12.3,-35.5,-54.6
4933,東京,8.2,16.3,13.2


## （９）予測誤差（H30推計2020 ÷ 国勢調査2020実績）


In [29]:
# 予測誤差: H30推計の2020予測 ÷ 国勢調査2020実績 → pop_err_2020_pred_2018_{半径}（%, 表記B）
#   式: (予測 / 実績 − 1) × 100  ＝ (pop_pred_2018_2020 / pop_2020 − 1) × 100
#   符号: ＋過大予測 / −過小予測（実績が分母）。信頼度は2km以上が高く、500mは参考。
_errcols = {}
for R in RADII:
    pred = station_group[f"pop_pred_2018_2020_{RNAME[R]}"].to_numpy(dtype="float64", na_value=np.nan)  # H30の2020予測
    act  = station_group[f"pop_2020_{RNAME[R]}"].to_numpy(dtype="float64", na_value=np.nan)            # 国勢調査2020実績(分母)
    with np.errstate(divide="ignore", invalid="ignore"):
        _errcols[f"pop_err_2020_pred_2018_{RNAME[R]}"] = np.round(np.where(act > 0, (pred / act - 1.0) * 100.0, np.nan), 1)
station_group = pd.concat([station_group, pd.DataFrame(_errcols, index=station_group.index)], axis=1)  # 断片化回避のため一括結合
_er = [c for c in station_group.columns if c.startswith("pop_err_2020_pred_2018_")]
e5 = station_group["pop_err_2020_pred_2018_5km"].to_numpy("float64")
e5 = e5[~np.isnan(e5)]
print(f"予測誤差 列数: {len(_er)}（{len(RADII)}半径）  "
      f"全国5km → 中央値 {np.median(e5):+.1f}% / MAPE {np.mean(np.abs(e5)):.1f}% / 過小予測 {(e5 < 0).mean() * 100:.0f}%")
# サニティ: 代表駅 × 全半径（＋過大 / −過小）
station_group[station_group["grp"].isin(["東京#0", "新宿#0", "二月田#0"])][
    ["駅名"] + [f"pop_err_2020_pred_2018_{RNAME[R]}" for R in RADII]]

予測誤差 列数: 6（6半径）  全国5km → 中央値 -0.6% / MAPE 2.4% / 過小予測 60%


,駅名,pop_err_2020_pred_2018_500m,pop_err_2020_pred_2018_1km,pop_err_2020_pred_2018_2km,pop_err_2020_pred_2018_5km,pop_err_2020_pred_2018_10km,pop_err_2020_pred_2018_20km
982,二月田,-21.9,-12.4,-5.7,-0.6,1.2,1.2
4376,新宿,158.9,9.5,-0.4,-1.8,-2.8,-2.2
4933,東京,182.5,47.1,-7.3,-2.5,-2.5,-2.4


## （10）信頼性フラグ（率の分母＝基準人口の大小）


In [30]:
# 信頼性フラグ: 率の分母＝基準人口 pop_{year}_{radius} が小さいと増減率・予測誤差が不安定
#   （例 和歌山大学前 base=1人→+388,100%／福島避難区域 2015≈0人）。データは正しい実態だが「参考値」と区別する。
#   pop_lowbase_{year}_{radius} = 1（base < 50人, 参考扱い） / 0（信頼）。各率は「分母の年×半径」のフラグを参照:
#     pop_gr_{新}_{旧}_{R}        → pop_lowbase_{旧}_{R}
#     pop_gr_pred_2024_{年}_{R}   → pop_lowbase_2020_{R}
#     pop_err_2020_pred_2018_{R}  → pop_lowbase_2020_{R}
LOWBASE_T = 50
_flag = {}
for year in [1995, 2000, 2005, 2010, 2015, 2020]:
    for R in RADII:
        base = station_group[f"pop_{year}_{RNAME[R]}"].to_numpy(dtype="float64")  # 実績人口はNaNなし
        _flag[f"pop_lowbase_{year}_{RNAME[R]}"] = (base < LOWBASE_T).astype("int8")
station_group = pd.concat([station_group, pd.DataFrame(_flag, index=station_group.index)], axis=1)  # 断片化回避
_lb = [c for c in station_group.columns if c.startswith("pop_lowbase_")]
print(f"信頼性フラグ 列数: {len(_lb)}（6年 × {len(RADII)}半径, 閾値 base<{LOWBASE_T}人）")
print("基準年別 low-base 駅数（半径別。≥5kmでほぼ消え20kmは0＝大半径は安定）:")
for year in [2020, 2015, 1995]:
    print(f"  {year}: " + "  ".join(f"{RNAME[R]}={int(station_group[f'pop_lowbase_{year}_{RNAME[R]}'].sum())}" for R in RADII))
# サニティ: 新興/災害駅は1、通常駅(浦和)は0
station_group[station_group["駅名"].isin(["和歌山大学前", "双葉", "浦和"])][
    ["駅名", "pop_2000_1km", "pop_lowbase_2000_1km", "pop_2015_10km", "pop_lowbase_2015_10km", "pop_lowbase_2020_5km"]]

信頼性フラグ 列数: 36（6年 × 6半径, 閾値 base<50人）
基準年別 low-base 駅数（半径別。≥5kmでほぼ消え20kmは0＝大半径は安定）:
  2020: 500m=468  1km=198  2km=91  5km=12  10km=3  20km=0
  2015: 500m=417  1km=179  2km=79  5km=15  10km=6  20km=0
  1995: 500m=341  1km=120  2km=52  5km=5  10km=0  20km=0


,駅名,pop_2000_1km,pop_lowbase_2000_1km,pop_2015_10km,pop_lowbase_2015_10km,pop_lowbase_2020_5km
2214,双葉,3507,0,1,1,0
2382,和歌山大学前,1,1,337174,0,0
5930,浦和,36636,0,2470225,0,0


## （11）地価公示（駅×半径の地価・各年推移）

`docs/land_price.md` §9 準拠。2026ファイルの内包系列（1983–2026）から、駅グループの **地価特徴量** を生成する。中央値・最寄点・各年水準パネル（グラフ用・開始2007）・増減率（no-match・年ペア）。採用 n≥1／n=0→Null、低信頼フラグ `lp_n<5`。

In [31]:
# （11）地価公示（L01・2026内包系列）→ 駅×半径の地価特徴量。docs/land_price.md §9 準拠
import zipfile, time
from pyproj import Transformer
from scipy.spatial import cKDTree

LP_ZIP = BASE_DIR / "data" / "地価公示" / "L01-26_GML.zip"
def _lpcol(y): return f"L01_{62 + (y - 1983):03d}"          # 1983=L01_062 .. 2026=L01_105
LP_PANEL_YEARS = list(range(2007, 2027))                    # 各年水準（グラフ・開始2007）
LP_GR_BASES    = [2025, 2023, 2021, 2016, 2011]             # 1y/3y/5y/10y/15y（2026基準）
LP_RADII       = [500, 1000, 2000, 5000, 10000]             # 水準・パネル
LP_GR_RADII    = [1000, 2000, 5000, 10000]                  # 増減率（500mは1点比でノイズ大→除外）
LP_RN          = {500: "500m", 1000: "1km", 2000: "2km", 5000: "5km", 10000: "10km"}
USE_MAP        = {"000": "住宅地", "003": "宅地見込地", "005": "商業地", "009": "工業地", "013": "調整区域内宅地等"}

with zipfile.ZipFile(LP_ZIP) as z:
    _shp = next(n for n in z.namelist() if n.lower().endswith(".shp"))
_need = ["L01_002", "L01_008"] + [_lpcol(y) for y in LP_PANEL_YEARS]
lp = gpd.read_file(f"/vsizip/{LP_ZIP}/{_shp}", columns=_need)   # 2026・EPSG:6668（数値/ASCII列のみ＝文字コード非依存）
if lp.crs is None: lp = lp.set_crs("EPSG:6668")

PYmat = np.column_stack([pd.to_numeric(lp[_lpcol(y)], errors="coerce").fillna(0).to_numpy(float) for y in LP_PANEL_YEARS])
Y2I = {y: i for i, y in enumerate(LP_PANEL_YEARS)}
p2026 = pd.to_numeric(lp["L01_008"], errors="coerce").to_numpy(float)            # 当年価格(=L01_105)
use_lab = pd.Series(lp["L01_002"].astype(str)).map(USE_MAP).fillna(lp["L01_002"].astype(str)).to_numpy()

lpx, lpy = tf_aea.transform(lp.geometry.x.to_numpy(), lp.geometry.y.to_numpy())  # 点・駅とも 6668→Albers
lp_tree = cKDTree(np.c_[lpx, lpy])
sx, sy = tf_aea.transform(station_group["lon"].to_numpy(), station_group["lat"].to_numpy())
nST = len(station_group)
print(f"地価2026: {len(lp):,}点  CRS={lp.crs}  パネル年={LP_PANEL_YEARS[0]}–{LP_PANEL_YEARS[-1]}")

# 最寄点（20km以内・k=1。20km内に無ければNaN）
dN, iN = lp_tree.query(np.c_[sx, sy], k=1, distance_upper_bound=20000)
okN = np.isfinite(dN); safeN = np.where(okN, iN, 0)
lp_near_price = np.where(okN, p2026[safeN], np.nan)
lp_near_dist  = np.where(okN, dN, np.nan)
lp_near_use   = np.where(okN, use_lab[safeN], None)

# 半径内 各年 median/点数（最大10km球を1回→半径マスク）。0＝その年は標準地でない→除外
panel_med = {y: {R: np.full(nST, np.nan) for R in LP_RADII} for y in LP_PANEL_YEARS}
panel_n   = {y: {R: np.zeros(nST, int)  for R in LP_RADII} for y in LP_PANEL_YEARS}
_ball = lp_tree.query_ball_point(np.c_[sx, sy], max(LP_RADII)); _t0 = time.time()
for j in range(nST):
    cand = np.asarray(_ball[j], dtype=int)
    if cand.size == 0: continue
    d = np.hypot(lpx[cand] - sx[j], lpy[cand] - sy[j]); sub_all = PYmat[cand]
    for R in LP_RADII:
        m = d <= R
        if not m.any(): continue
        sub = sub_all[m]
        for y in LP_PANEL_YEARS:
            col = sub[:, Y2I[y]]; pos = col[col > 0]
            if pos.size:
                panel_n[y][R][j] = pos.size; panel_med[y][R][j] = np.median(pos)
    if (j + 1) % 3000 == 0: print(f"  {j+1}/{nST} ({time.time()-_t0:.0f}s)")
print(f"  半径集計完了 {time.time()-_t0:.0f}s")

# スカラー特徴を station_group に付与（lp_ 接頭辞）。採用 n≥1／n=0→NaN、フラグ lp_n<5
_lp = {"lp_near_price": pd.Series(np.round(lp_near_price)).astype("Int64"),
       "lp_near_dist_m": pd.Series(np.round(lp_near_dist)).astype("Int64"),
       "lp_near_use": lp_near_use}
for R in LP_RADII:
    rn = LP_RN[R]
    _lp[f"lp_med_{rn}"]  = pd.Series(np.round(panel_med[2026][R])).astype("Int64")
    _lp[f"lp_n_{rn}"]    = panel_n[2026][R].astype("int64")
    _lp[f"lp_lown_{rn}"] = (panel_n[2026][R] < 5).astype("int8")
for base in LP_GR_BASES:
    for R in LP_GR_RADII:
        rn = LP_RN[R]; new = panel_med[2026][R]; old = panel_med[base][R]
        with np.errstate(divide="ignore", invalid="ignore"):
            _lp[f"lp_gr_2026_{base}_{rn}"] = np.round(np.where((old > 0) & (new > 0), (new / old - 1.0) * 100.0, np.nan), 1)
        _lp[f"lp_gr_lown_2026_{base}_{rn}"] = (np.minimum(panel_n[2026][R], panel_n[base][R]) < 5).astype("int8")
station_group = pd.concat([station_group, pd.DataFrame(_lp, index=station_group.index)], axis=1)  # 断片化回避

# 各年水準パネル（グラフ用・ロング形式・n≥1のみ）→ 別CSV（保存セル）
_grp = station_group["grp"].to_numpy(); _parts = []
for y in LP_PANEL_YEARS:
    for R in LP_RADII:
        n = panel_n[y][R]; keep = n > 0
        if keep.any():
            _parts.append(pd.DataFrame({"grp": _grp[keep], "year": y, "radius": LP_RN[R],
                                        "lp_med": np.round(panel_med[y][R][keep]).astype("int64"),
                                        "lp_n": n[keep].astype("int64")}))
lp_panel = pd.concat(_parts, ignore_index=True)

_lpc = [c for c in station_group.columns if c.startswith("lp_")]
print(f"地価スカラー列: {len(_lpc)}（最寄3＋水準/点数/フラグ15＋増減率/フラグ40）  各年水準パネル行(n≥1): {len(lp_panel):,}")
# サニティ: 都心(高)・地方(低)
station_group[station_group["駅名"].isin(["東京", "大阪", "二月田"])][
    ["駅名", "lp_near_price", "lp_near_use", "lp_med_2km", "lp_n_2km", "lp_med_5km", "lp_gr_2026_2016_5km", "lp_gr_2026_2011_5km"]]

地価2026: 25,565点  CRS=EPSG:6668  パネル年=2007–2026


  3000/9273 (2s)


  6000/9273 (5s)


  9000/9273 (7s)


  半径集計完了 7s
地価スカラー列: 58（最寄3＋水準/点数/フラグ15＋増減率/フラグ40）  各年水準パネル行(n≥1): 674,105


,駅名,lp_near_price,lp_near_use,lp_med_2km,lp_n_2km,lp_med_5km,lp_gr_2026_2016_5km,lp_gr_2026_2011_5km
982,二月田,16500,住宅地,16500,1,21350,-20.9,-36.8
2956,大阪,24700000,商業地,1770000,51,884500,102.9,119.5
4933,東京,37600000,商業地,5540000,84,2820000,115.3,104.3


In [32]:
# CSV 出力: DB(Supabase/PostGIS)取込を見据えた最終成果物。
#   - lon/lat を保持（geometry は取込時に ST_SetSRID(ST_MakePoint(lon,lat),4326) で生成）。
#   - 日本語の列【名】を ASCII snake_case に変換（値は日本語のまま）→ PostgREST/API・SQLで扱いやすい。
OUT_DIR = BASE_DIR / "data" / "derived"
OUT_DIR.mkdir(parents=True, exist_ok=True)
COLMAP = {"駅名": "station_name", "都道府県": "prefecture", "運営会社": "operator"}  # rename は該当列のみ適用

# メイン: 1群1行 × 全データ（乗降客・人口実績/将来推計/増減率/予測誤差/信頼性フラグ）
station_dataset = station_group.rename(columns=COLMAP)
station_dataset.to_csv(OUT_DIR / "station_dataset.csv", index=False)

# 監査明細: 1群×1社の素データ（各年値・在/欠測・参照年）。検証・トレース用
station_operator_detail = station_group_operator.rename(columns=COLMAP)
station_operator_detail.to_csv(OUT_DIR / "station_operator_detail.csv", index=False)

# 地価 各年水準パネル（グラフ用・ロング形式。station_dataset とは grp で結合）
lp_panel.to_csv(OUT_DIR / "station_landprice_yearly.csv", index=False)

_na_main = [c for c in station_dataset.columns if not c.isascii()]
_na_det  = [c for c in station_operator_detail.columns if not c.isascii()]
print("saved:")
print(f"  メインデータセット : {OUT_DIR/'station_dataset.csv'}  {station_dataset.shape}")
print(f"  監査明細(1群×1社)  : {OUT_DIR/'station_operator_detail.csv'}  {station_operator_detail.shape}")
print(f"  地価 各年水準パネル: {OUT_DIR/'station_landprice_yearly.csv'}  {lp_panel.shape}")
print(f"  ASCII化した列名     : {COLMAP}")
print(f"  残存する非ASCII列名 : main={_na_main} / detail={_na_det}（期待: 両方 []）")

saved:
  メインデータセット : /Users/kensuke26/Desktop/AI_Native/01Project/dev_Database_Map/AI-Database-Map/data/derived/station_dataset.csv  (9273, 397)
  監査明細(1群×1社)  : /Users/kensuke26/Desktop/AI_Native/01Project/dev_Database_Map/AI-Database-Map/data/derived/station_operator_detail.csv  (9781, 32)
  地価 各年水準パネル: /Users/kensuke26/Desktop/AI_Native/01Project/dev_Database_Map/AI-Database-Map/data/derived/station_landprice_yearly.csv  (674105, 5)
  ASCII化した列名     : {'駅名': 'station_name', '都道府県': 'prefecture', '運営会社': 'operator'}
  残存する非ASCII列名 : main=[] / detail=[]（期待: 両方 []）
